# ATPESC 2026 micro-lab: tokens → shifted targets → generation

**Time:** 6–8 minutes  
**Goal:** inspect how text becomes IDs, then train a tiny causal model using the same next-token objective as a decoder-only LLM.

Everything is local. After the environment is installed, this notebook performs no downloads and needs no network access.

## Setup — run this cell first

The helper module (`lab_core.py`) contains the deterministic setup, the transparent tokenizers, the one-block Transformer, batching, and the training loop. Run this cell, then work through the cells below.

In [ ]:
import collections
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from lab_core import (
    SEED,
    build_demo_tokenizers,
    build_training_tokenizer,
    configure_determinism,
    count_parameters,
    encode_repeated_corpus,
    format_cost_comparison,
    format_encoding,
    load_corpus,
    sample_text,
    train_tiny_model,
)

configure_determinism(SEED)
print(
    f"Python {platform.python_version()} | PyTorch {torch.__version__} | device=cpu | seed={SEED}"
)

## Part A — Tokenization (about 2 minutes)

These are deliberately transparent teaching tokenizers:

- **raw Unicode code points**: one ID per code point;
- **UTF-8 bytes**: one ID per byte;
- **toy HPC subwords**: deterministic longest match over a small, hand-selected vocabulary.

The toy vocabulary is **not** claimed to be BPE, WordPiece, Unigram, or a production LLM tokenizer.

In [ ]:
tokenizers = build_demo_tokenizers()

for text in ("MPI_Comm_rank returns 0.", " Aurora trains LLMs."):
    print(f"\ntext: {text!r}")
    print(format_encoding(tokenizers["raw_codepoints"].encode(text)))
    print(format_encoding(tokenizers["toy_subwords"].encode(text)))

**Try first:** Before running, predict where `_`, punctuation, and the leading space will land. The four-token HPC result is favorable to this hand-selected vocabulary — shorter is not automatically better.

### Normalization comes first: NFC

The same character can be stored as different code points: `é` may be **one** code point (`U+00E9`) or **two** (`e` + combining acute accent `U+0301`). They look identical on screen but are different data, so a naive tokenizer would assign them different IDs.

**NFC** (Unicode Normalization Form C) composes such sequences into one canonical form. Our training tokenizer applies NFC **first**, then assigns one token per code point — so both spellings of `é` map to the *same* ID. (Siblings: **NFD** decomposes; **NFKC/NFKD** also fold compatibility variants like `ﬁ`→`fi`, which can change meaning, so NFC is the safe default.)

Watch the next cell: the **raw** code-point tokenizer sees 1 vs 2 tokens, while **NFC** collapses both to a single `é` token.

In [ ]:
# These strings look alike, but the second uses e + COMBINING ACUTE ACCENT.
for text in ("é", "e\u0301"):
    raw = tokenizers["raw_codepoints"].encode(text)
    nfc = tokenizers["nfc_codepoints"].encode(text)
    print(f"\ntext: {text!r}")
    print(format_encoding(raw))
    print(format_encoding(nfc))

In [ ]:
cost_text = "MPI_Comm_rank(comm, &rank); // ε = 1.0e−12"
print(
    format_cost_comparison(
        cost_text,
        (
            tokenizers["utf8_bytes"],
            tokenizers["raw_codepoints"],
            tokenizers["toy_subwords"],
        ),
    )
)

**Interpretation:** tokenization sets sequence length $T$. For dense attention, the score matrix contains $T^2$ position pairs. That is a useful mechanism-level cost proxy—not a complete wall-clock performance model.

## Part B — Shift, train, sample (about 4–5 minutes)

For the model we use NFC-normalized Unicode code points. This keeps the vocabulary and output layer tiny. Production LLMs commonly use learned subwords or bytes, but the causal objective does not depend on that choice.

In [ ]:
corpus = load_corpus(Path("data/tiny_corpus.txt"))
training_tokenizer = build_training_tokenizer(corpus)

example = training_tokenizer.encode("aurora trains")
ids = torch.tensor(example.ids, dtype=torch.long)
inputs = ids[:-1]
targets = ids[1:]

print("tokens:    ", list(example.tokens))
print("input IDs: ", inputs.tolist())
print("target IDs:", targets.tolist())

**Try first:** Predict `targets[0]` before running. Every target is exactly one position ahead of its input; the causal mask prevents the model from seeing future tokens.

### Visualize: what the model trains on

The corpus is small and repetitive, so a few characters dominate. The model mostly learns these local patterns.

In [ ]:
counts = collections.Counter(c for c in corpus if c.strip())
items = counts.most_common()
chars = [c for c, _ in items]
freqs = [n for _, n in items]
plt.figure(figsize=(7, 3))
plt.bar(chars, freqs, color="#118ACB")
plt.ylabel("count")
plt.title("Character frequencies in the training corpus")
plt.tight_layout()
plt.show()

In [ ]:
token_stream = encode_repeated_corpus(corpus, training_tokenizer)
result = train_tiny_model(token_stream, vocab_size=training_tokenizer.vocab_size)

print(f"parameters: {count_parameters(result.model):,}")
for step, loss in sorted(result.losses.items()):
    print(f"step {step:>3}: evaluation loss = {loss:.4f}")
print(f"CPU training runtime: {result.elapsed_seconds:.3f} s")

**Try first:** Predict how the loss will change before running. The reported loss uses one fixed batch, while training uses deterministic sampled batches. Lower loss means better fit to this corpus — not truth or general capability.

### Visualize: the training loss

Each point is the fixed-batch evaluation loss. It falls steeply, then flattens as the tiny model fits the small corpus.

In [ ]:
steps = sorted(result.losses)
plt.figure(figsize=(7, 3.5))
plt.plot(steps, [result.losses[s] for s in steps], marker="o", color="#0061AF")
plt.xlabel("training step")
plt.ylabel("next-token loss")
plt.title("Training loss decreases as the model learns")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
prompt = "aurora "
sample = sample_text(result.model, training_tokenizer, prompt=prompt)
print("prompt:", repr(prompt))
print("seeded top-k sample:", repr(sample))

### Visualize: next-token probabilities

Given a prompt, the model outputs a probability for **every** token in the vocabulary. Here are the most likely next characters — the distribution the sampler draws from.

In [ ]:
probe = "aurora trains "
probe_ids = torch.tensor([training_tokenizer.encode(probe).ids])
with torch.no_grad():
    logits = result.model(probe_ids)[0, -1]
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, 8)
labels = [repr(training_tokenizer.id_to_token[i]) for i in top.indices.tolist()]
plt.figure(figsize=(7, 3.5))
plt.bar(labels, top.values.tolist(), color="#009F90")
plt.ylabel("probability")
plt.title(f"P(next token | {probe!r})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Debrief

1. Tokenization defined the prediction units and sequence length.
2. Shifting defined each next-token target.
3. Cross-entropy and parameter updates changed next-token probabilities.
4. Fixed-weight generation repeatedly sampled and appended one token.

> **This is not an LLM.** It has one Transformer block, about ten thousand parameters, a character-level vocabulary, and a tiny repeated corpus. It nevertheless uses the same shifted next-token objective as the lecture's decoder-only running case.

